# Practical 20: Multiple Object Tracking (MOT) with **YOLOv11** + **Kalman** + **Hungarian**


In this practical, you will build a **Multiple Object Tracking (MOT)** system by combining:
- **Detector:** YOLOv11 (we use YOLOv11; auto‑fallback to YOLOv8 if needed)
- **Tracker core:** **Kalman filter** (motion prediction) + **Hungarian** assignment (ID matching)
- **Two variants to compare:**
  1. **ByteTrack‑style** (IoU‑driven, two‑stage) using the `supervision` library
  2. **Mini DeepSORT‑style** (educational), adds a lightweight **appearance cue** (HSV histogram) to motion

---
**Student Note:** Try to connect this step with earlier practicals. Think about: What problem does this solve? Why not use only YOLO without tracking?


### Objectives
- Understand the **detector → tracker** pipeline for MOT.
- Run **YOLOv11** detections frame‑by‑frame.
- Maintain identities over time with **Kalman + Hungarian**.
- Compare **ByteTrack** vs a **DeepSORT‑style** approach with appearance cues.
- Save an **annotated tracking video** and analyze **ID switches**.


### Connections & Prerequisites
- You explored CNNs and object detection earlier (e.g., Practical #18 on YOLO).
- From Practical #6 (Evaluation Metrics), recall precision/recall; here you’ll reason about **ID switches** and **tracking stability**.


## 0) Setup (Install & Imports)

If you’re offline, ensure the packages are pre‑installed and skip the `pip` cell.


In [ ]:

%%bash
set -e
python -V
# Core CV & tracking stack (Py3.11 friendly)
pip install -q -U ultralytics supervision opencv-python scipy filterpy
# Optional speed-up lib used by some trackers; safe to ignore if it fails
pip install -q lapx || true


### What this code does
This code block runs an important step in the MOT pipeline. Pay attention to:
- **Input**: What data/frame is passed into the model
- **Process**: Which algorithm (YOLOv11, DeepSORT, ByteTrack) is being applied
- **Output**: What new information do we get (bounding boxes, IDs, tracks)

Run the cell and then carefully observe the printed/logged results.

In [ ]:

import os, time, math, sys
from dataclasses import dataclass

import numpy as np
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO
import supervision as sv

from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

print("Imports successful")
os.makedirs("outputs", exist_ok=True)


Imports successful


In [ ]:

# Quick environment checks (optional)
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("Torch not available (CPU-only). That's fine for this practical.")


Torch: 2.5.1
CUDA available: True



## 1) Data: bring a short test video

We’ll try to download a small public clip; if that fails, we’ll **create a synthetic video** so the notebook always runs.


In [ ]:

def create_synthetic_video(path="synthetic_demo.mp4", W=640, H=360, frames=200):
    """Create a synthetic clip with moving rectangles (works fully offline)."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter(path, fourcc, 30.0, (W, H))
    rng = np.random.default_rng(42)
    N = 6
    objs = []
    for i in range(N):
        x = rng.integers(50, W-150); y = rng.integers(30, H-120)
        w = rng.integers(40, 80);    h = rng.integers(40, 80)
        vx = rng.integers(-3, 4);    vy = rng.integers(-3, 4)
        objs.append([x,y,w,h,vx,vy])
    for t in range(frames):
        frame = np.full((H,W,3), 235, np.uint8)
        cv2.putText(frame, "SYNTHETIC DEMO (no real objects)", (10,25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (70,70,70), 2)
        for i,(x,y,w,h,vx,vy) in enumerate(objs):
            if x<=0 or x+w>=W: vx = -vx
            if y<=0 or y+h>=H: vy = -vy
            x += vx; y += vy
            objs[i] = [x,y,w,h,vx,vy]
            cv2.rectangle(frame, (x,y), (x+w,y+h), (0,50+30*i,150), 2)
        vw.write(frame)
    vw.release()
    return path

def prepare_demo_video():
    """Try to download a tiny real traffic clip; otherwise use a synthetic fallback."""
    demo_path = "data/videos/traffic.mp4"
    os.makedirs(os.path.dirname(demo_path), exist_ok=True)
    if not os.path.exists(demo_path):
        try:
            import urllib.request
            url = "https://media.roboflow.com/video_examples/los-angeles-traffic-short.mp4"
            print("Downloading demo video...")
            urllib.request.urlretrieve(url, demo_path)
            print("Downloaded:", demo_path)
            return demo_path
        except Exception as e:
            print("Download failed:", e)
    if os.path.exists(demo_path):
        return demo_path
    print("🎬 Creating synthetic demo video...")
    return create_synthetic_video()

video_path = prepare_demo_video()
print("Using video:", video_path)


Using video: data/videos/traffic.mp4



## 2) Detector: YOLOv11 (with fallback to YOLOv8)

We’ll try `yolo11n.pt` (nano) → `yolo11s.pt` → fallback to `yolov8n.pt` if needed.


In [ ]:

def setup_yolo_detector():
    model_weights = ["yolo11n.pt", "yolo11s.pt", "yolov8n.pt"]
    last_error = None
    for w in model_weights:
        try:
            print(f"Loading {w} ...")
            m = YOLO(w)
            print(f"Loaded: {w}")
            return m
        except Exception as e:
            last_error = e
            print(f"Failed: {w} ({e})")
    raise RuntimeError(f"Could not load any YOLO weights from {model_weights}. "
                       f"Place weights locally or check internet. Last error: {last_error}")

model = setup_yolo_detector()

# Tracking config (tweak for your scene)
TRACK_CLASS_IDS = [0, 2, 3, 5, 7]  # person, car, motorcycle, bus, truck
CONF_THRES = 0.25
IOU_THRES_NMS = 0.45


Loading yolo11n.pt ...
Loaded: yolo11n.pt


In [ ]:

# (Optional) test: run YOLO on a single frame to confirm everything works
cap = cv2.VideoCapture(video_path)
ok, frame = cap.read()
cap.release()
if ok:
    res = model.predict(frame, conf=CONF_THRES, iou=IOU_THRES_NMS, verbose=False)[0]
    print("Detections:", len(res.boxes) if res.boxes is not None else 0)
else:
    print("Could not read a frame from the video.")


Detections: 13



## 3) Tracker A : ByteTrack‑style (via `supervision`)

**Idea:** match high‑confidence detections first, then recover matches for low‑confidence ones.

**Per frame:**
1. YOLO → boxes, scores, classes  
2. Convert to `sv.Detections`, filter classes  
3. `ByteTrack.update_with_detections(...)` → get IDs  
4. Draw boxes/labels and optional motion trails


In [ ]:

def track_with_bytetrack(video_path, output_path="outputs/track_bytetrack.mp4",
                         target_classes=TRACK_CLASS_IDS, conf_thresh=CONF_THRES, iou_thresh=IOU_THRES_NMS):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    # Robust init: try new parameter names, else fallback to old ones (for different library versions)
    try:
        tracker = sv.ByteTrack(
            track_activation_threshold=0.25,
            lost_track_buffer=30,
            minimum_matching_threshold=0.8,
            frame_rate=fps
        )
    except TypeError:
        tracker = sv.ByteTrack(
            track_thresh=0.25,
            track_buffer=30,
            match_thresh=0.8
        )

    box_annotator   = sv.BoxAnnotator()
    trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=int(fps*2))

    t0 = time.time(); frames = 0
    print("Tracking with ByteTrack...")
    while True:
        ok, frame = cap.read()
        if not ok: break

        res = model.predict(frame, conf=conf_thresh, iou=iou_thresh, verbose=False)[0]
        dets = sv.Detections.from_ultralytics(res)
        dets = dets[np.isin(dets.class_id, target_classes)]

        tracks = tracker.update_with_detections(dets)

        labels = []
        names = res.names
        for i in range(len(tracks)):
            tid  = tracks.tracker_id[i] if tracks.tracker_id is not None else None
            cid  = int(tracks.class_id[i]) if tracks.class_id is not None else -1
            conf = float(tracks.confidence[i]) if tracks.confidence is not None else 0.0
            name = names.get(cid, str(cid)) if hasattr(names, "get") else str(cid)
            labels.append(f"#{tid if tid is not None else '?'} {name} {conf:.2f}")

        frame_ = trace_annotator.annotate(scene=frame.copy(), detections=tracks)
        frame_ = box_annotator.annotate(scene=frame_, detections=tracks)

        writer.write(frame_)
        frames += 1

    cap.release(); writer.release()
    print(f"Saved: {output_path}  |  frames={frames}  |  time={time.time()-t0:.1f}s")

track_with_bytetrack(video_path, output_path='outputs/track_bytetrack.mp4')


Tracking with ByteTrack...
Saved: outputs/track_bytetrack.mp4  |  frames=150  |  time=2.8s



## 4) Tracker B : Mini DeepSORT‑style

**DeepSORT = SORT (Kalman + Hungarian) + Appearance (ReID).**  
For teaching, we use a simple **HSV histogram** (no heavy ReID net).

**Design:**
- **State:** ([cx, cy, a, r, v<sub>cx</sub>, v<sub>cy</sub>, v<sub>a</sub>]), where **a = w · h**, **r = w / h**
- **Predict:** Kalman (constant velocity)
- **Cost:** (1 − IoU) + λ · (1 − cosine_sim(hist))
- **Assign:** Hungarian (min-cost) + IoU gating



In [ ]:

# ---------- Geometry ----------
def xyxy_to_center_area_ratio(box):
    x1,y1,x2,y2 = box
    w = max(0.0, x2-x1); h = max(0.0, y2-y1)
    cx = x1 + w/2.0; cy = y1 + h/2.0
    a = w*h; r = w/(h+1e-6)
    return np.array([cx, cy, a, r], dtype=np.float32)

def center_area_ratio_to_xyxy(state):
    cx,cy,a,r = state
    w = math.sqrt(max(0.0, a*r))
    h = max(1e-6, w/(r+1e-12))
    x1 = cx - w/2.0; y1 = cy - h/2.0
    x2 = cx + w/2.0; y2 = cy + h/2.0
    return np.array([x1,y1,x2,y2], dtype=np.float32)

def calculate_iou(a, b):
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    ix1,iy1 = max(ax1,bx1), max(ay1,by1)
    ix2,iy2 = min(ax2,bx2), min(ay2,by2)
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw*ih
    ua = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter + 1e-6
    return inter/ua

def cosine_sim(a, b, eps=1e-8):
    an = a/(np.linalg.norm(a)+eps)
    bn = b/(np.linalg.norm(b)+eps)
    return float(np.clip(np.dot(an, bn), -1.0, 1.0))

# ---------- Appearance (HSV hist) ----------
def extract_hsv_histogram(img, box, bins=(8,8,8)):
    x1,y1,x2,y2 = [int(v) for v in box]
    H,W = img.shape[:2]
    x1 = max(0, min(W-1, x1)); y1 = max(0, min(H-1, y1))
    x2 = max(0, min(W,   x2)); y2 = max(0, min(H,   y2))
    if x2<=x1 or y2<=y1:
        return np.zeros(sum(bins), dtype=np.float32)
    roi = img[y1:y2, x1:x2]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    h = cv2.calcHist([hsv],[0],None,[bins[0]],[0,180]).flatten()
    s = cv2.calcHist([hsv],[1],None,[bins[1]],[0,256]).flatten()
    v = cv2.calcHist([hsv],[2],None,[bins[2]],[0,256]).flatten()
    feat = np.concatenate([h,s,v]).astype(np.float32)
    feat /= (np.sum(feat)+1e-8)
    return feat

# ---------- Kalman filter ----------
def create_kalman_filter():
    kf = KalmanFilter(dim_x=7, dim_z=4)
    dt = 1.0
    kf.F = np.array([
        [1,0,0,0,dt,0, 0],
        [0,1,0,0,0, dt,0],
        [0,0,1,0,0, 0, dt],
        [0,0,0,1,0, 0, 0 ],
        [0,0,0,0,1, 0, 0 ],
        [0,0,0,0,0, 1, 0 ],
        [0,0,0,0,0, 0, 1 ],
    ], dtype=np.float32)
    kf.H = np.array([
        [1,0,0,0,0,0,0],
        [0,1,0,0,0,0,0],
        [0,0,1,0,0,0,0],
        [0,0,0,1,0,0,0],
    ], dtype=np.float32)
    kf.P *= 10.0
    kf.R *= 1.0
    kf.Q = np.eye(7, dtype=np.float32) * 0.01
    return kf

@dataclass
class TrackState:
    track_id: int
    kalman_filter: KalmanFilter
    bounding_box: np.ndarray
    class_id: int
    appearance_feature: np.ndarray
    age: int = 0
    time_since_update: int = 0
    hit_count: int = 1

class MiniDeepSORT:
    def __init__(self, iou_threshold=0.2, appearance_weight=0.25, max_age=30, min_hits=2):
        self.iou_threshold = iou_threshold
        self.appearance_weight = appearance_weight
        self.max_age = max_age
        self.min_hits = min_hits
        self.tracks = []
        self.next_track_id = 1

    def predict_all_tracks(self):
        for t in self.tracks:
            t.kalman_filter.predict()
            t.bounding_box = center_area_ratio_to_xyxy(t.kalman_filter.x[:4].flatten())
            t.age += 1; t.time_since_update += 1

    def update_with_detections(self, frame, det_boxes, det_classes):
        N, M = len(det_boxes), len(self.tracks)
        if M == 0 and N > 0:
            for i in range(N):
                self._create_new_track(frame, det_boxes[i], det_classes[i])
            return

        det_feats = [extract_hsv_histogram(frame, b) for b in det_boxes]
        C = np.zeros((M, N), dtype=np.float32)
        for m, tr in enumerate(self.tracks):
            for n in range(N):
                iou_cost = 1.0 - calculate_iou(tr.bounding_box, det_boxes[n])
                app_cost = 1.0 - cosine_sim(tr.appearance_feature, det_feats[n])
                C[m, n] = iou_cost + self.appearance_weight * app_cost

        row, col = linear_sum_assignment(C) if C.size>0 else (np.array([],int), np.array([],int))
        matched_tr, matched_det = set(), set()
        for r,c in zip(row, col):
            if calculate_iou(self.tracks[r].bounding_box, det_boxes[c]) >= self.iou_threshold:
                self._update_existing_track(self.tracks[r], frame, det_boxes[c], det_classes[c], det_feats[c])
                matched_tr.add(r); matched_det.add(c)

        for n in range(N):
            if n not in matched_det:
                self._create_new_track(frame, det_boxes[n], det_classes[n])

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]

    def _create_new_track(self, frame, box, cls_id):
        kf = create_kalman_filter()
        kf.x[:4] = xyxy_to_center_area_ratio(box).reshape(4,1)
        tr = TrackState(
            track_id=self.next_track_id,
            kalman_filter=kf,
            bounding_box=np.array(box, dtype=np.float32),
            class_id=int(cls_id),
            appearance_feature=extract_hsv_histogram(frame, box)
        )
        self.tracks.append(tr); self.next_track_id += 1

    def _update_existing_track(self, tr, frame, box, cls_id, feat):
        tr.kalman_filter.update(xyxy_to_center_area_ratio(box))
        tr.bounding_box = center_area_ratio_to_xyxy(tr.kalman_filter.x[:4].flatten())
        tr.appearance_feature = 0.7*tr.appearance_feature + 0.3*feat
        tr.time_since_update = 0; tr.hit_count += 1; tr.class_id = int(cls_id)

def track_with_mini_deepsort(video_path, output_path="outputs/track_minideepsort.mp4",
                             target_classes=TRACK_CLASS_IDS, conf_thresh=CONF_THRES, iou_thresh=IOU_THRES_NMS):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    tracker = MiniDeepSORT(iou_threshold=0.2, appearance_weight=0.25, max_age=30, min_hits=2)
    palette = {}

    t0 = time.time(); frames = 0
    print("Tracking with Mini DeepSORT‑style...")
    while True:
        ok, frame = cap.read()
        if not ok: break

        res = model.predict(frame, conf=conf_thresh, iou=iou_thresh, verbose=False)[0]
        det_boxes, det_cls = [], []
        if res.boxes is not None:
            for b, c in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy().astype(int)):
                if c in target_classes:
                    det_boxes.append(b.astype(np.float32)); det_cls.append(int(c))

        tracker.predict_all_tracks()
        tracker.update_with_detections(frame, det_boxes, det_cls)

        # Draw confirmed tracks only (reduce flicker)
        out = frame.copy()
        for tr in tracker.tracks:
            if tr.hit_count < tracker.min_hits and tr.age < tracker.min_hits:
                continue
            x1,y1,x2,y2 = [int(v) for v in tr.bounding_box]
            if tr.track_id not in palette:
                rng = np.random.default_rng(tr.track_id)
                palette[tr.track_id] = tuple(int(x) for x in rng.integers(40,255,size=3))
            color = palette[tr.track_id]
            cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)
            cv2.putText(out, f"#{tr.track_id}", (x1, max(0,y1-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        writer.write(out); frames += 1

    cap.release(); writer.release()
    print(f"Saved: {output_path}  |  frames={frames}  |  time={time.time()-t0:.1f}s")

track_with_mini_deepsort(video_path, output_path='outputs/track_minideepsort.mp4')


Tracking with Mini DeepSORT‑style...
Saved: outputs/track_minideepsort.mp4  |  frames=150  |  time=4.4s



### Tracker B : Accurate Counting (Line & Zone) *Improved*
We add **robust counting** to  Mini‑DeepSORT tracker:

- **Unique ID count**: number of distinct track IDs seen (no double counting).
- **Line crossing count**: count when a track **crosses a virtual line**, with **direction** (e.g., Up vs Down).
- **Zone entry count**: count when a track **enters** a polygon Region‑Of‑Interest (ROI) from outside.

**How it works (idea):**
- For each `track_id`, we store a small **state**: previous center point, last side of the line, and whether it was in the ROI.
- On each frame, we update the state and detect **events**: a change in side ⇒ a line crossing; `False → True` ROI state ⇒ an entry.
- We render live counters on the video.

> This design is resistant to jitter and ID flicker because counts are tied to **state transitions**, not raw positions.


In [ ]:
# ========================= Improved Counting Utilities =========================
def _center_of(box):
    x1,y1,x2,y2 = box
    return ((x1+x2)/2.0, (y1+y2)/2.0)

def _line_side(pt, a, b):
    """Return signed side of point `pt` relative to directed line a->b.
    >0 means on left side, <0 right side, =0 collinear.
    """
    (x,y) = pt; (x1,y1) = a; (x2,y2) = b
    return (x - x1)*(y2 - y1) - (y - y1)*(x2 - x1)

def _crosses_line(prev_pt, curr_pt, a, b, eps=1e-6):
    """Return (crossed:boolean, direction:int) where direction is +1 or -1.
    We detect a crossing when the sign of `_line_side` changes.
    Direction is determined by the movement projected onto the line's normal.
    """
    s1 = _line_side(prev_pt, a, b)
    s2 = _line_side(curr_pt, a, b)
    # If both very close to line, treat as no crossing to avoid jitter
    if abs(s1) < eps and abs(s2) < eps:
        return (False, 0)
    if s1 == 0: s1 = eps if s2 <= 0 else -eps  # nudge to opposite side so sgn change is captured once
    if s2 == 0: s2 = -eps if s1 > 0 else eps
    crossed = (s1 > 0 and s2 < 0) or (s1 < 0 and s2 > 0)
    if not crossed:
        return (False, 0)
    # Direction: use a normal vector n = (dy, -dx) and the motion vector
    dx, dy = (curr_pt[0]-prev_pt[0]), (curr_pt[1]-prev_pt[1])
    nx, ny = (b[1]-a[1], -(b[0]-a[0]))  # rotate (dx_line, dy_line) by +90°
    move_dot_n = dx*nx + dy*ny
    direction = 1 if move_dot_n > 0 else -1
    return (True, direction)

def _point_in_polygon(pt, polygon):
    """Ray casting algorithm for point in polygon. polygon: [(x,y), ...]"""
    x, y = pt
    inside = False
    n = len(polygon)
    if n < 3:
        return False
    x0, y0 = polygon[-1]
    for i in range(n):
        x1, y1 = polygon[i]
        # Check edge (x0,y0)->(x1,y1)
        cond = ((y1 > y) != (y0 > y)) and (x < (x0 - x1) * (y - y1) / (y0 - y1 + 1e-12) + x1)
        if cond: inside = not inside
        x0, y0 = x1, y1
    return inside


# ---------------- Pretty box drawing helpers ----------------
def _draw_label_with_bg(img, text, org, font_scale=0.6, thickness=2, text_color=(255,255,255), bg_color=(0,0,0)):
    (w,h), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
    x,y = org
    cv2.rectangle(img, (x, y-h-baseline-4), (x+w+6, y+2), bg_color, -1)
    cv2.putText(img, text, (x+3, y-baseline-2), cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, thickness)

def _draw_corners(img, box, color=(50,200,50), thickness=2, frac=0.25):
    x1,y1,x2,y2 = [int(v) for v in box]
    w = max(0, x2-x1); h = max(0, y2-y1)
    lw = max(1, int(w*frac)); lh = max(1, int(h*frac))
    # top-left
    cv2.line(img, (x1,y1), (x1+lw,y1), color, thickness)
    cv2.line(img, (x1,y1), (x1,y1+lh), color, thickness)
    # top-right
    cv2.line(img, (x2,y1), (x2-lw,y1), color, thickness)
    cv2.line(img, (x2,y1), (x2,y1+lh), color, thickness)
    # bottom-left
    cv2.line(img, (x1,y2), (x1+lw,y2), color, thickness)
    cv2.line(img, (x1,y2), (x1,y2-lh), color, thickness)
    # bottom-right
    cv2.line(img, (x2,y2), (x2-lw,y2), color, thickness)
    cv2.line(img, (x2,y2), (x2,y2-lh), color, thickness)

def _draw_translucent_box(img, box, color=(50,200,50), alpha=0.25, thickness=2):
    x1,y1,x2,y2 = [int(v) for v in box]
    overlay = img.copy()
    cv2.rectangle(overlay, (x1,y1), (x2,y2), color, -1)
    cv2.addWeighted(overlay, alpha, img, 1-alpha, 0, img)
    cv2.rectangle(img, (x1,y1), (x2,y2), color, thickness)

# ========================= Enhanced Mini DeepSORT with Counting =========================
def track_with_mini_deepsort_v2(
    video_path,
    output_path="outputs/track_minideepsort_counted.mp4",
    target_classes=TRACK_CLASS_IDS,
    conf_thresh=CONF_THRES,
    iou_thresh=IOU_THRES_NMS,
    count_mode="line",                   # "unique", "line", or "zone"
    line=((50, 200), (600, 200)),        # default horizontal line (y=200) for direction up/down
    zone_polygon=None                    # e.g., [(x1,y1),(x2,y2),(x3,y3),(x4,y4)]
):
    """
    Mini DeepSORT-style tracker with stable counting.
    - count_mode="unique": counts unique track IDs (ever seen)
    - count_mode="line": counts line crossings with direction (+1 / -1)
    - count_mode="zone": counts entries into a polygon ROI

    Returns: dict with counts summary.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # If no line or zone provided, choose sensible defaults (center line / middle box)
    if count_mode == "line" and line is None:
        line = ((0, H//2), (W, H//2))
    if count_mode == "zone" and zone_polygon is None:
        margin = int(min(W,H)*0.15)
        zone_polygon = [(margin, margin), (W-margin, margin), (W-margin, H-margin), (margin, H-margin)]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    model = setup_yolo_detector()
    tracker = MiniDeepSORT(max_age=20, min_hits=3, appearance_weight=0.3)  # slightly more stable

    # Counting state
    seen_ids = set()                 # for unique counting
    meta = {}                        # track_id -> {"prev_center":(x,y), "last_side":int, "in_zone":bool}
    line_up = line_down = 0
    zone_entries = 0

    t0 = time.time()
    frames = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # 1) Run detector
        res = model.predict(frame, conf=conf_thresh, iou=iou_thresh, verbose=False)[0]
        boxes = []
        classes = []
        if res.boxes is not None and len(res.boxes) > 0:
            for b in res.boxes:
                cls_id = int(b.cls.item()) if hasattr(b.cls, "item") else int(b.cls)
                if target_classes and (cls_id not in target_classes):
                    continue
                x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].tolist()]
                boxes.append([x1,y1,x2,y2])
                classes.append(cls_id)

        # 2) Update tracker
        tracker.update_with_detections(frame, boxes, classes)

        # 3) Counting logic
        # Iterate only confirmed tracks to reduce flicker
        for tr in tracker.tracks:
            if tr.hit_count < tracker.min_hits and tr.age < tracker.min_hits:
                continue
            tid = tr.track_id
            seen_ids.add(tid)

            c = _center_of(tr.bounding_box.tolist())

            if tid not in meta:
                # Initialize per-track meta
                last_side = 0
                if count_mode == "line":
                    last_side = 1 if _line_side(c, line[0], line[1]) > 0 else -1
                in_zone = False
                if count_mode == "zone":
                    in_zone = _point_in_polygon(c, zone_polygon)
                meta[tid] = {"prev_center": c, "last_side": last_side, "in_zone": in_zone}
                continue  # need a previous center for next frame

            prev_c = meta[tid]["prev_center"]

            if count_mode == "line":
                # Check for side change → crossing
                crossed, dirn = _crosses_line(prev_c, c, line[0], line[1])
                if crossed:
                    if dirn > 0:
                        line_up += 1
                    else:
                        line_down += 1
                    # After counting, reset last_side to current side to prevent double counting
                    meta[tid]["last_side"] = 1 if _line_side(c, line[0], line[1]) > 0 else -1

            elif count_mode == "zone":
                was_in = meta[tid]["in_zone"]
                now_in = _point_in_polygon(c, zone_polygon)
                if (not was_in) and now_in:
                    zone_entries += 1
                meta[tid]["in_zone"] = now_in

            # update previous center
            meta[tid]["prev_center"] = c

        # 4) Draw visualization
        out = frame.copy()

        # Draw line or zone if used
        if count_mode == "line":
            a, b = line
            cv2.line(out, (int(a[0]), int(a[1])), (int(b[0]), int(b[1])), (0,255,255), 2)
        elif count_mode == "zone":
            pts = np.array(zone_polygon, dtype=np.int32).reshape(-1,1,2)
            cv2.polylines(out, [pts], isClosed=True, color=(0,255,255), thickness=2)


        # 4) Draw visualization
        out = frame.copy()

        # Draw line or zone if used
        if count_mode == "line":
            a, b = line
            cv2.line(out, (int(a[0]), int(a[1])), (int(b[0]), int(b[1])), (0,255,180), 2)
        elif count_mode == "zone":
            pts = np.array(zone_polygon, dtype=np.int32).reshape(-1,1,2)
            cv2.polylines(out, [pts], isClosed=True, color=(0,255,180), thickness=2)

        # Draw tracks (pretty, less crowded)
        for tr in tracker.tracks:
            if tr.hit_count < tracker.min_hits and tr.age < tracker.min_hits:
                continue
            x1,y1,x2,y2 = [int(v) for v in tr.bounding_box]
            area = max(1, (x2-x1)*(y2-y1))

            # thickness scales with box size; skip labels for tiny boxes
            t = 1 if area < 12_000 else (2 if area < 45_000 else 3)
            show_label = area >= 8000

            # choose calm green-ish color, translucent fill + corner ticks
            _draw_translucent_box(out, (x1,y1,x2,y2), color=(60,180,120), alpha=0.18, thickness=t)
            _draw_corners(out, (x1,y1,x2,y2), color=(60,180,120), thickness=t, frac=0.22)

            if show_label:
                _draw_label_with_bg(out, f"#{tr.track_id}", (x1, max(18, y1-6)),
                                    font_scale=0.6, thickness=2,
                                    text_color=(255,255,255), bg_color=(30,110,80))

        # Draw counters (top-left HUD)
        y0 = 28
        _draw_label_with_bg(out, f"Unique IDs: {len(seen_ids)}", (10, y0))
        if count_mode == "line":
            _draw_label_with_bg(out, f"Line crossings  Up:+{line_up}  Down:-{line_down}", (10, y0+32))
        elif count_mode == "zone":
            _draw_label_with_bg(out, f"Zone entries: {zone_entries}", (10, y0+32))

        writer.write(out)
        frames += 1


    cap.release(); writer.release()
    summary = {
        "unique_ids": len(seen_ids),
        "line_up": line_up,
        "line_down": line_down,
        "zone_entries": zone_entries,
        "frames": frames,
        "time_s": round(time.time()-t0, 2)
    }
    print("Summary:", summary)
    print(f"Saved: {output_path}")
    return summary

# Example run (line counting across the vertical middle):
# Feel free to adjust `line` or switch `count_mode` to "zone" with a polygon.
_ = track_with_mini_deepsort_v2(
    video_path,
    output_path="outputs/track_minideepsort_counted.mp4",
    count_mode="line",
    line=None  # auto: horizontal line at mid‑height
)


Loading yolo11n.pt ...
Loaded: yolo11n.pt
Summary: {'unique_ids': 61, 'line_up': 0, 'line_down': 1, 'zone_entries': 0, 'frames': 150, 'time_s': 7.35}
Saved: outputs/track_minideepsort_counted.mp4



## 5) Experiments (Do These)

1. **Track only people**: set `TRACK_CLASS_IDS = [0]` and re‑run both trackers.  
2. **Confidence trade‑off**: try `CONF_THRES = 0.1` vs `0.5`.  
3. **Appearance weight** (Mini DeepSORT): change `appearance_weight` to `0.4`.  
4. **Speed/Quality**: swap `yolo11n.pt` ↔ `yolo11s.pt`.  



## 7) What Next

- **Next practical: Classic Image Segmentation**→ thresholding (global/Otsu/adaptive), edges→contours, region growing/Watershed, k-means, with morphology clean-up and IoU/Dice checks.  

---
📝 This practical was designed to gives you a a compact, hands-on intro to Multiple Object Tracking with YOLOv11 + Kalman + Hungarian.